# ⚡ Módulo 11 - Notebook 01: PySpark Core y SparkSession

## 🚀 Introducción a Apache Spark y Computación Distribuida

**Libro:** Saliendo de lo Pandito  
**Módulo:** 11 - PySpark Core SparkSession  
**Duración estimada:** 75 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** la arquitectura distribuida de Spark  
✅ **Crear** SparkSession y configurar Spark  
✅ **Trabajar** con Spark DataFrames  
✅ **Distinguir** entre Pandas y Spark DataFrames  
✅ **Aplicar** transformaciones básicas distribuidas

---

## 📋 Pre-requisitos

* ✅ Módulos 03-10 completados (Pandas, GeoPandas, H3)
* ✅ Conocimiento sólido de Pandas DataFrames
* ✅ Familiaridad con conceptos de Big Data

---

## 📚 Contenido

1. ¿Qué es Apache Spark?
2. Arquitectura de Spark (Driver, Executors, Cluster)
3. SparkSession: El Punto de Entrada
4. Spark DataFrames vs Pandas DataFrames
5. Creación de Spark DataFrames
6. Caso Integrador: Primer Análisis Distribuido

---

## 💡 Por qué importa

**Spark transforma el análisis de datos:**

* 📊 **Escala:** De GB a TB/PB sin cambiar código
* ⚡ **Velocidad:** Procesamiento paralelo en cluster
* 🌐 **Distribución:** Datos divididos en múltiples máquinas
* 💪 **Tolerancia a fallos:** Reconstrucción automática

**El estándar de la industria para Big Data**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Verificar que SparkSession esté disponible
    print(f"\n⚡ SparkSession disponible: {spark is not None}")
    print(f"   Version de Spark: {spark.version}")
    print(f"   Application ID: {spark.sparkContext.applicationId}")
    print(f"   Master: {spark.sparkContext.master}")
    
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df_spark = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    # Información del Spark DataFrame
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df_spark.count():,}")
    print(f"   🏛️ Particiones: {df_spark.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    print(f"\n📋 Esquema del Spark DataFrame:")
    df_spark.printSchema()
    
    print(f"\n🎯 Este notebook trabajará con Spark DataFrames DISTRIBUIDOS")
    print(f"   (A diferencia de Pandas que corre en una sola máquina)")
    
    print(f"\n💡 Nota: En Databricks, SparkSession ya está creado automáticamente")
    print(f"   como la variable 'spark'")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_spark = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Apache Spark: Computación Distribuida

### ⚡ ¿Qué es Apache Spark?

**Apache Spark** es un motor de procesamiento distribuido de datos a gran escala.

**Concepto clave:**
* Pandas: 1 máquina, 1 CPU, RAM limitada
* Spark: N máquinas, N CPUs, RAM distribuida

**Analogía:**
```
Pandas = Un chef cocinando solo
Spark  = 100 chefs cocinando en paralelo
```

---

### 🏗️ Arquitectura de Spark

**Componentes principales:**

```
┌─────────────────────────────────────────┐
│           DRIVER PROGRAM                │
│        (Tu código Python)               │
│      ┌──────────────────┐               │
│      │  SparkSession    │               │
│      │   (spark)        │               │
│      └──────────────────┘               │
└─────────────────┬───────────────────────┘
                  │
        ┌─────────┴─────────┐
        │                   │
   ┌────▼─────┐       ┌────▼─────┐
   │ EXECUTOR │       │ EXECUTOR │
   │ (Worker) │       │ (Worker) │
   │  Task 1  │       │  Task 3  │
   │  Task 2  │       │  Task 4  │
   └──────────┘       └──────────┘
```

**1️⃣ Driver:**
* Tu código Python
* Coordina el trabajo
* Crea el plan de ejecución

**2️⃣ Executors (Workers):**
* Máquinas que procesan datos
* Cada uno tiene su CPU y RAM
* Ejecutan tareas en paralelo

**3️⃣ SparkSession:**
* Punto de entrada a Spark
* Configura la conexión al cluster
* Variable `spark` en Databricks

---

### 🔧 SparkSession: El Punto de Entrada

**En Databricks:**
```python
# Ya está creado automáticamente
spark  # Variable global
```

**En Python local (Jupyter, Colab):**
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiApp") \
    .master("local[*]") \
    .getOrCreate()
```

**Configuraciones comunes:**
```python
spark = SparkSession.builder \
    .appName("Ventas") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
```

---

### 📊 Spark DataFrame vs Pandas DataFrame

| Característica | Pandas DataFrame | Spark DataFrame |
|----------------|------------------|------------------|
| **Tamaño** | Hasta RAM (GB) | Ilimitado (TB/PB) |
| **Ejecución** | Una máquina | Cluster distribuido |
| **Velocidad (small data)** | ⚡ Rápido | 🐢 Overhead |
| **Velocidad (big data)** | ❌ No cabe | ⚡⚡⚡ Paralelo |
| **API** | Eager (inmediata) | Lazy (perezosa) |
| **Tipos de datos** | NumPy types | Spark types |

**Regla de oro:**
* **< 10 GB:** Usa Pandas
* **> 10 GB:** Usa Spark

---

### 🔄 Conversión Pandas ↔ Spark

**Pandas → Spark:**
```python
pandas_df = pd.DataFrame({'col': [1, 2, 3]})
spark_df = spark.createDataFrame(pandas_df)
```

**Spark → Pandas:**
```python
spark_df = spark.table("mi_tabla")
pandas_df = spark_df.toPandas()  # ⚠️ Trae TODO a memoria
```

⚠️ **CUIDADO:** `toPandas()` trae TODO el DataFrame a la memoria del driver. Solo usar con datos pequeños.

---

### 🛠️ Creación de Spark DataFrames

**1️⃣ Desde lista de tuplas:**
```python
data = [("Juan", 30), ("María", 25)]
df = spark.createDataFrame(data, ["nombre", "edad"])
```

**2️⃣ Desde Pandas:**
```python
pd_df = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
df = spark.createDataFrame(pd_df)
```

**3️⃣ Desde tabla:**
```python
df = spark.table("catalog.schema.tabla")
```

**4️⃣ Desde archivo:**
```python
df = spark.read.parquet("/path/to/file.parquet")
df = spark.read.csv("/path/to/file.csv", header=True, inferSchema=True)
```

---

### 💡 Conceptos Clave

**Inmutabilidad:**
* Spark DataFrames son INMUTABLES
* Cada transformación crea un NUEVO DataFrame

```python
df1 = spark.table("ventas")
df2 = df1.filter("ventas > 1000")  # df1 NO cambia, df2 es nuevo
```

**Lazy Evaluation:**
* Transformaciones NO se ejecutan inmediatamente
* Solo cuando hay una ACCIÓN (count, show, collect)

```python
df2 = df1.filter(...)  # No ejecuta nada aún
df3 = df2.select(...)  # Tampoco
df3.count()            # AHORA ejecuta todo
```

---

### 📈 Casos de Uso

**Cuándo usar Spark:**
* ✅ Datos > 10 GB
* ✅ Procesamiento ETL de producción
* ✅ Datos en Datalake (S3, ADLS, GCS)
* ✅ Pipelines de datos automatizados

**Cuándo NO usar Spark:**
* ❌ Análisis exploratorio pequeño
* ❌ Prototipado rápido
* ❌ Datos < 1 GB
* ❌ Operaciones que requieren ordenamiento global

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg
import warnings
warnings.filterwarnings('ignore')

print("⚡ PYSPARK CORE: SparkSession y DataFrames Distribuidos")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

# En Databricks, 'spark' ya está disponible
try:
    print(f"Versión de Spark: {spark.version}")
    print(f"SparkSession ID: {spark.sparkContext.applicationId}")
except:
    print("⚠️  SparkSession no disponible (ejecuta en Databricks)")

print("\n🎯 En este notebook aprenderás:")
print("  • Arquitectura de Spark (Driver, Executors)")
print("  • SparkSession: Punto de entrada")
print("  • Spark DataFrame vs Pandas DataFrame")
print("  • Creación de Spark DataFrames")

print("\n📖 Métodos clave:")
print("  - spark.table('catalog.schema.table')")
print("  - spark.createDataFrame(data, schema)")
print("  - df.show()  # Acción")
print("  - df.count()  # Acción")
print("  - df.toPandas()  # Convertir a Pandas")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# En Databricks la SparkSession 'spark' ya está inicializada por defecto.
# Código demostrativo de creación de DataFrame en PySpark:

datos_pyspark = [
    ("Transaccion_1", "Sucursal_A", 15000.0),
    ("Transaccion_2", "Sucursal_B", 22000.0),
    ("Transaccion_3", "Sucursal_A", 35000.0)
]

columnas = ["ID_Transaccion", "Sucursal", "Monto"]

print("Demostración PySpark Core en Databricks Free Edition")
print(f"Registros a procesar en clúster distribuido: {len(datos_pyspark)}")

